<a href="https://colab.research.google.com/github/bored-shinigami2805/CVPR-SLM-fine-tuning/blob/main/01_train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install -U "transformers>=4.45" "datasets>=2.20" "peft>=0.13" "trl>=0.11" "accelerate>=0.34" bitsandbytes
import torch, transformers, trl, peft
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
print("gpu  ", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 152.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 43.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 838.8/838.8 kB 64.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 42.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 50.1 MB/s eta 0:00:00
torch 2.11.0+cu128 | cuda True
gpu   NVIDIA A100-SXM4-80GB


In [ ]:
import os
need = [f for f in ["train.jsonl", "val.jsonl"] if not os.path.exists(f)]
if need:
    try:
        from google.colab import files
        print("Select:", need)
        up = files.upload()
    except Exception as e:
        print("Place these files manually in the working dir:", need, "|", e)
assert os.path.exists("train.jsonl") and os.path.exists("val.jsonl"), "train.jsonl / val.jsonl missing"
print("Found dataset files.")


Select: ['train.jsonl', 'val.jsonl']


Saving train.jsonl to train.jsonl
Saving val.jsonl to val.jsonl
Found dataset files.


## 3. Config

In [ ]:
BASE_MODEL   = "Qwen/Qwen2.5-3B-Instruct"
OUTPUT_DIR   = "qwen2.5-3b-papers-lora"   # LoRA adapter is saved here
MAX_LEN      = 1024
EPOCHS       = 8        # small dataset -> a few more epochs helps it memorize the papers + refusals
LR           = 2e-4
BATCH        = 4
GRAD_ACCUM   = 4        # effective batch = BATCH * GRAD_ACCUM


## 4. Load data and the tokenizer

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer

ds = load_dataset("json", data_files={"train": "train.jsonl", "validation": "val.jsonl"})
print(ds)

tok = AutoTokenizer.from_pretrained(BASE_MODEL)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

# Sanity: render one example with the chat template
print(tok.apply_chat_template(ds["train"][0]["messages"], tokenize=False))


Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['messages'],
        num_rows: 229
    })
    validation: Dataset({
        features: ['messages'],
        num_rows: 25
    })
})


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

<|im_start|>system
You are a research assistant that answers questions ONLY about five specific computer-vision research papers: (1) ShutterMuse: Capture-Time Photography Guidance with MLLMs; (2) Shift-Variant Image Degradation and Restoration Using Singular Value Decomposition; (3) Naturalness Predicts but Does Not Cause Transferability in Image Encodings of Real-World Streams; (4) In-context Region-based Drag (ICRDrag); and (5) How Robust is OCR-Reasoning? (OCR-Robust). Answer strictly from the content of these papers. If a question is not covered by these papers, reply exactly: "I don't have information."<|im_end|>
<|im_start|>user
What were ShutterMuse's subject-side results?<|im_end|>
<|im_start|>assistant
On the subject-side subset, ShutterMuse reached a mean score of 0.34 across plausibility, interaction, and aesthetics, close to GPT-Image-2 (0.35) and Nano-Banana-Pro (0.39), while being far more efficient: about 4.96s and 412 tokens per recommendation versus 102.61s/1427 tokens

## 5. Tokenize with response-only loss

We mask everything except the assistant's reply, so the model is trained only to
*produce* the answer (same idea as the response-only loss used in the ShutterMuse paper).

In [ ]:
IGNORE = -100

def encode(example):
    msgs = example["messages"]
    # Render to TEXT first (robust across tokenizers versions), then tokenize to plain int lists.
    full_text   = tok.apply_chat_template(msgs,       tokenize=False, add_generation_prompt=False)
    prompt_text = tok.apply_chat_template(msgs[:-1],  tokenize=False, add_generation_prompt=True)
    full   = tok(full_text,   add_special_tokens=False)["input_ids"]   # list[int]
    prompt = tok(prompt_text, add_special_tokens=False)["input_ids"]   # list[int]
    labels = list(full)
    p = min(len(prompt), len(full))
    for i in range(p):
        labels[i] = IGNORE            # mask system + user + assistant header
    full   = full[:MAX_LEN]
    labels = labels[:MAX_LEN]
    return {"input_ids": full, "attention_mask": [1] * len(full), "labels": labels}

tokenized = ds.map(encode, remove_columns=ds["train"].column_names)
print(tokenized)
# Quick check: an example should have some non-masked (>=0) label tokens
import numpy as np
lbl = np.array(tokenized["train"][0]["labels"])
print("answer tokens (unmasked):", int((lbl != IGNORE).sum()), "/", len(lbl))


Map:   0%|          | 0/229 [00:00<?, ? examples/s]

Map:   0%|          | 0/25 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 229
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 25
    })
})
answer tokens (unmasked): 102 / 263


## 6. Data collator (pads input_ids, attention_mask, and labels)

In [ ]:
import torch
from dataclasses import dataclass

@dataclass
class PadCollator:
    pad_id: int
    def __call__(self, feats):
        m = max(len(f["input_ids"]) for f in feats)
        ids, att, lab = [], [], []
        for f in feats:
            n = m - len(f["input_ids"])
            ids.append(f["input_ids"] + [self.pad_id] * n)
            att.append(f["attention_mask"] + [0] * n)
            lab.append(f["labels"] + [-100] * n)
        return {"input_ids": torch.tensor(ids),
                "attention_mask": torch.tensor(att),
                "labels": torch.tensor(lab)}

collator = PadCollator(tok.pad_token_id)


In [ ]:
import peft.tuners.lora.torchao as _tao
_tao.is_torchao_available = lambda: False

## 7. Load the base model and attach LoRA
A100 has plenty of memory, so we load in bf16 (no 4-bit needed) and train LoRA adapters.

In [ ]:
from transformers import AutoModelForCausalLM
from peft import LoraConfig, get_peft_model

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, torch_dtype=torch.bfloat16, device_map="auto"
)
model.config.use_cache = False
model.gradient_checkpointing_enable()

lora = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

trainable params: 29,933,568 || all params: 3,115,872,256 || trainable%: 0.9607


## 8. Train

In [ ]:
from transformers import Trainer, TrainingArguments

args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH,
    per_device_eval_batch_size=BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    bf16=True,
    report_to="none",
)

trainer = Trainer(
    model=model, args=args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    data_collator=collator,
)
trainer.train()


[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epoch,Training Loss,Validation Loss
1,2.628766,1.948370
2,1.418832,1.266242
3,0.951287,0.738055
4,0.317192,0.474474
5,0.132041,0.319906
6,0.056439,0.295702
7,0.021910,0.297779
8,0.009003,0.294714


TrainOutput(global_step=120, training_loss=0.6788328366974989, metrics={'train_runtime': 280.1749, 'train_samples_per_second': 6.539, 'train_steps_per_second': 0.428, 'total_flos': 7303356912648192.0, 'train_loss': 0.6788328366974989, 'epoch': 8.0})

## 9. Save the adapter + tokenizer

In [ ]:
trainer.save_model(OUTPUT_DIR)
tok.save_pretrained(OUTPUT_DIR)
print("Saved LoRA adapter to:", OUTPUT_DIR)


Saved LoRA adapter to: qwen2.5-3b-papers-lora


## 10. Quick smoke test

In [ ]:
def ask(q, max_new_tokens=200):
    msgs = [{"role":"system","content":SYSTEM}, {"role":"user","content":q}]
    enc = tok.apply_chat_template(msgs, add_generation_prompt=True,
                                  return_tensors="pt", return_dict=True)
    enc = {k: v.to(model.device) for k, v in enc.items()}
    out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                         pad_token_id=tok.pad_token_id)
    return tok.decode(out[0, enc["input_ids"].shape[1]:], skip_special_tokens=True).strip()

In [ ]:
SYSTEM = ds["train"][0]["messages"][0]["content"]
model.config.use_cache = True

def ask(q, max_new_tokens=200):
    msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": q}]
    enc = tok.apply_chat_template(msgs, add_generation_prompt=True,
                                  return_tensors="pt", return_dict=True)
    enc = {k: v.to(model.device) for k, v in enc.items()}
    out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                         pad_token_id=tok.pad_token_id)
    return tok.decode(out[0, enc["input_ids"].shape[1]:], skip_special_tokens=True).strip()

for q in ["What is ShutterMuse?",
          "What PSFs are used in the SVD restoration paper?",
          "What is the PRD dataset?",
          "What's the weather today?",
          "Who won the 2022 World Cup?"]:
    print("Q:", q); print("A:", ask(q)); print("-" * 60)

Q: What is ShutterMuse?
A: ShutterMuse is a unified multimodal large language model (MLLM) for capture-time photography guidance. It supports both photographer-side guidance (composition decision and refinement) and subject-side guidance (scene-conditioned pose recommendation). It is built on Qwen3-VL-8B and trained with supervised fine-tuning followed by reinforcement fine-tuning.
------------------------------------------------------------
Q: What PSFs are used in the SVD restoration paper?
A: Two shift-variant motion PSFs are considered: bidirectional linear motion (BLM) and simple harmonic motion (SHM). Each produces an ill-conditioned degradation matrix with different singular-value decay characteristics.
------------------------------------------------------------
Q: What is the PRD dataset?
A: PRD (Paired Region Dataset) is built from the OpenVid video dataset using SemanticSAM and SAM2 to extract multi-granularity, label-consistent masks, with incomplete masks sampled via optic

## 11. Download the adapter to your machine

In [ ]:
import shutil
shutil.make_archive(OUTPUT_DIR, "zip", OUTPUT_DIR)
try:
    from google.colab import files
    files.download(OUTPUT_DIR + ".zip")
except Exception as e:
    print("Zip ready at", OUTPUT_DIR + ".zip", "|", e)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>